In [6]:
import os
import cohere
from groq import Groq
from qdrant_client import QdrantClient

co_client = cohere.Client()
groq_client = Groq()

## Embedding Fucntion

In [7]:
def get_embedding(text, model="embed-english-v3.0"):
    response = co_client.embed(
        texts=[text],
        model=model,
        input_type="search_query"
    )
    return response.embeddings[0]

## Retrieval Function

In [8]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [9]:
def retrieve_data(query,qdrant_client, k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="amazon-items-collection-00",
        query=query_embedding,
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_contexts = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_contexts.append(result.payload.get("description"))
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])
    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_contexts": retrieved_contexts,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [10]:
retrieved_context = retrieve_data("what kind of wireless headphones can I get?", qdrant_client, k=10)

In [11]:
retrieved_context

{'retrieved_context_ids': ['B0BZRCY2H9',
  'B0BRXDR9GF',
  'B0C5B3JHH7',
  'B0C3HC6S6R',
  'B0BYCMG5BC',
  'B0B6H8ZD1P',
  'B0BZL6B7GB',
  'B0BN32HS1G',
  'B0BZPC1SM5',
  'B0BJV7Y7P7'],
 'retrieved_contexts': ["STADOR Wireless Earbuds Bluetooth V5.3 in-Ear Headphones 56H Playtime Stereo Earphones with LED Power Display Charging Case IPX7 Waterproof Ear Buds with Microphones for Work Fitness Sports, Black Crystal Clear Sound Quality - These wireless headphones features with large 13mm delivers to deliver rich, immersive sound with deep bass and clear treble. Enjoy your favorite tunes like never before with our STADOR Bluetooth wireless earbuds and ensure that you hear every note and beat with utmost clarity, whether you're commuting or working out. Long Battery Life & LED Power Display: The earphones come with 400 mAh large capacity charging case, which can provide extra 3 times full charge for earbuds (Total of 56 hours of battery life). Up to 7 hours of listening time on a single char

## Format retrieved context function 

In [12]:
def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(
        context["retrieved_context_ids"],
        context["retrieved_contexts"],
        context["retrieved_context_ratings"]
    ):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

In [13]:
preprocessed_context = process_context(retrieved_context)

In [14]:
print(preprocessed_context)

- ID: B0BZRCY2H9, rating: 4.2, description: STADOR Wireless Earbuds Bluetooth V5.3 in-Ear Headphones 56H Playtime Stereo Earphones with LED Power Display Charging Case IPX7 Waterproof Ear Buds with Microphones for Work Fitness Sports, Black Crystal Clear Sound Quality - These wireless headphones features with large 13mm delivers to deliver rich, immersive sound with deep bass and clear treble. Enjoy your favorite tunes like never before with our STADOR Bluetooth wireless earbuds and ensure that you hear every note and beat with utmost clarity, whether you're commuting or working out. Long Battery Life & LED Power Display: The earphones come with 400 mAh large capacity charging case, which can provide extra 3 times full charge for earbuds (Total of 56 hours of battery life). Up to 7 hours of listening time on a single charge, ensuring that your headphones are always ready when you need them. The case features a dual digital power display to show the charging status of the earbuds and th

## Create Prompt Funtion 

In [15]:
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
{preprocessed_context}

Question:
{question}
"""

    return prompt

In [16]:
prompt = build_prompt(preprocessed_context, "what kind of wireless headphones can I get?")

In [17]:
print(prompt)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
- ID: B0BZRCY2H9, rating: 4.2, description: STADOR Wireless Earbuds Bluetooth V5.3 in-Ear Headphones 56H Playtime Stereo Earphones with LED Power Display Charging Case IPX7 Waterproof Ear Buds with Microphones for Work Fitness Sports, Black Crystal Clear Sound Quality - These wireless headphones features with large 13mm delivers to deliver rich, immersive sound with deep bass and clear treble. Enjoy your favorite tunes like never before with our STADOR Bluetooth wireless earbuds and ensure that you hear every note and beat with utmost clarity, whether you're commuting or working out. Long Battery Life & LED Power Display: The earphones come with 400 mAh large capacity charging case, which c

## Generate answer function

In [18]:
def generate_answer(prompt):
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [19]:
answer = generate_answer(prompt)
print(answer)

Based on the available products, you can consider the following wireless headphones:

1. STADOR Wireless Earbuds (ID: B0BZRCY2H9) - These earbuds feature Bluetooth V5.3, stereo sound, and a long battery life of 56 hours.
2. SoundPEATS Air Conduction Headphones (ID: B0BRXDR9GF) - These over-ear headphones feature bass boost, 17 hours of battery life, and sweatproof design.
3. Skullcandy Hesh Evo Bluetooth Headphones (ID: B0C5B3JHH7) - These headphones feature 36 hours of battery life, clear blue sound, and a foldable design.

These three options are wireless headphones available in the market, but the best choice would depend on your specific needs and preferences.


## Combined RAG Pipeline 

In [20]:
def rag_pipeline(question, top_k=5):

    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return answer

In [21]:
print(rag_pipeline("what kind of wireless headphones can I get with ratings above 4?"))

Based on the available products, you can consider the following wireless headphones with ratings above 4:

1. Skullcandy Hesh Evo Bluetooth Headphones (ID: B0C5B3JHH7, rating: 4.6) - These headphones offer big, premium sound without the big, premium price tag and come with features like Tile finding technology, 36 hours of battery life, and rapid charge capability.
2. STADOR Wireless Earbuds (ID: B0BZRCY2H9, rating: 4.2) - This product features crystal-clear sound quality, long battery life, and a compact charging case with LED power display. It also has Bluetooth 5.3 technology for stable and fast connections.

Both of these options meet your criteria and can provide you with high-quality sound and features.
